In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "e662579a-7cf4-4800-9ab8-fae6381625a9"
RESOURCE_GROUP = "rg-mlab1-seerat24"
WORKSPACE = "mlw-lab1-seerat24"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)


Connected to: mlw-lab1-seerat24


In [2]:
ENDPOINT_NAME = "knn-maint-seerat-ok"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created successfully:", ENDPOINT_NAME)

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.16.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Endpoint created successfully: knn-maint-seerat-ok


In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# Direct 100% of incoming traffic to the blue deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready!")
